In [1]:
import pandas as pd

covid_data = pd.read_csv('data/covid_data.csv')
display(covid_data.head())

vaccinations_data = pd.read_csv('data/country_vaccinations.csv')
vaccinations_data = vaccinations_data[
    ['country', 'date', 'total_vaccinations', 
     'people_vaccinated', 'people_vaccinated_per_hundred',
     'people_fully_vaccinated', 'people_fully_vaccinated_per_hundred',
     'daily_vaccinations', 'vaccines']
]
display(vaccinations_data)

,date,province/state,country,confirmed,deaths,recovered
0,01/22/2020,Anhui,China,1.0,0.0,0.0
1,01/22/2020,Beijing,China,14.0,0.0,0.0
2,01/22/2020,Chongqing,China,6.0,0.0,0.0
3,01/22/2020,Fujian,China,1.0,0.0,0.0
4,01/22/2020,Gansu,China,0.0,0.0,0.0


,country,date,total_vaccinations,people_vaccinated,people_vaccinated_per_hundred,people_fully_vaccinated,people_fully_vaccinated_per_hundred,daily_vaccinations,vaccines
0,Afghanistan,2021-02-22,0.0,0.0,0.00,NaN,NaN,NaN,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi..."
1,Afghanistan,2021-02-23,NaN,NaN,NaN,NaN,NaN,1367.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi..."
2,Afghanistan,2021-02-24,NaN,NaN,NaN,NaN,NaN,1367.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi..."
3,Afghanistan,2021-02-25,NaN,NaN,NaN,NaN,NaN,1367.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi..."
4,Afghanistan,2021-02-26,NaN,NaN,NaN,NaN,NaN,1367.0,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi..."
...,...,...,...,...,...,...,...,...,...
42790,Zimbabwe,2021-09-01,4270430.0,2615233.0,17.33,1655197.0,10.97,36416.0,"Oxford/AstraZeneca, Sinopharm/Beijing, Sinovac..."
42791,Zimbabwe,2021-09-02,4323735.0,2649505.0,17.56,1674230.0,11.09,39711.0,"Oxford/AstraZeneca, Sinopharm/Beijing, Sinovac..."
42792,Zimbabwe,2021-09-03,4372216.0,2681657.0,17.77,1690559.0,11.20,42317.0,"Oxford/AstraZeneca, Sinopharm/Beijing, Sinovac..."
42793,Zimbabwe,2021-09-04,4400246.0,2698332.0,17.88,1701914.0,11.28,41413.0,"Oxford/AstraZeneca, Sinopharm/Beijing, Sinovac..."


Что нужно сделать:

1. В таблице covid_data рассчитать суммарное ежедневное число заболевших во всех провинциях/штатах в каждой стране;

2. В таблицах не совпадает число стран, а иногда и их названия (исправить). При объединении таблиц по столбцу мы теряем данные (в данной задаче незначительно). Избежать этого можно ручными преобразованиями данных — искать различия в названиях стран в таблицах и преобразовывать их.

3. Таблицы имеют разный период наблюдений (вакцины появились позже, чем вирус). Объединив данные через inner, мы можем потерять большое количество наблюдений в таблице covid_data.

In [2]:
# Группируем таблицу по дате и названию страны, и рассчитываем суммарные показатели по всем
# регионам. То есть переходим от данных по регионам к данным по странам
covid_data = covid_data.groupby(
    ['date', 'country'],
    as_index=False
)[['confirmed', 'deaths', 'recovered']].sum()

In [3]:
# Преобразуем даты в формат datetime
covid_data['date'] = pd.to_datetime(covid_data['date'])

In [4]:
# Создадим признак больных на данный момент (active). Для этого вычтем из общего числа
# зафиксированных случаев число смертей и число выздоровевших пациентов
covid_data['active'] = covid_data['confirmed'] - covid_data['deaths'] - covid_data['recovered']

In [5]:
# Создадим признак ежедневного прироста числа заболевших, умерших и выздоровевших людей.
# Для этого отсортируем данные по названию страны, а затем по датам. После этого произведем
# группировку по странам и рассчитаем разницу между «вчера и сегодня» с помощью метода diff()
covid_data = covid_data.sort_values(by=['country', 'date'])
covid_data['daily_confirmed'] = covid_data.groupby('country')['confirmed'].diff()
covid_data['daily_deaths'] = covid_data.groupby('country')['deaths'].diff()
covid_data['daily_recovered'] = covid_data.groupby('country')['recovered'].diff()

In [6]:
display(covid_data)

,date,country,confirmed,deaths,recovered,active,daily_confirmed,daily_deaths,daily_recovered
11337,2020-02-24,Afghanistan,1.0,0.0,0.0,1.0,NaN,NaN,NaN
11570,2020-02-25,Afghanistan,1.0,0.0,0.0,1.0,0.0,0.0,0.0
11807,2020-02-26,Afghanistan,1.0,0.0,0.0,1.0,0.0,0.0,0.0
12051,2020-02-27,Afghanistan,1.0,0.0,0.0,1.0,0.0,0.0,0.0
12299,2020-02-28,Afghanistan,1.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
15933,2020-03-12,occupied Palestinian territory,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16595,2020-03-14,occupied Palestinian territory,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16938,2020-03-15,occupied Palestinian territory,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17290,2020-03-16,occupied Palestinian territory,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
vaccinations_data['date'] = pd.to_datetime(vaccinations_data['date'])

In [8]:
covid_data.describe()

,date,confirmed,deaths,recovered,active,daily_confirmed,daily_deaths,daily_recovered
count,86785,8.678500e+04,86785.000000,8.678500e+04,8.678500e+04,86564.000000,86564.000000,8.656400e+04
mean,2020-10-15 20:19:34.931151616,3.024953e+05,7190.332627,1.780289e+05,1.172760e+05,1963.295492,40.820618,1.237704e+03
min,2020-01-22 00:00:00,0.000000e+00,0.000000,0.000000e+00,-1.955600e+04,-348667.000000,-5337.000000,-6.399531e+06
25%,2020-06-25 00:00:00,8.120000e+02,11.000000,2.940000e+02,1.100000e+02,1.000000,0.000000,0.000000e+00
50%,2020-10-17 00:00:00,9.598000e+03,158.000000,5.313000e+03,1.516000e+03,55.000000,1.000000,1.600000e+01
75%,2021-02-07 00:00:00,9.872700e+04,1864.000000,6.439900e+04,1.450900e+04,622.000000,10.000000,3.520000e+02
max,2021-05-29 00:00:00,3.325194e+07,594306.000000,2.545432e+07,3.265763e+07,823225.000000,4561.000000,1.123456e+06
std,NaN,1.632725e+06,32713.193900,8.792410e+05,1.268258e+06,11895.703910,198.349234,2.411494e+04


In [9]:
vaccinations_data.describe()

,date,total_vaccinations,people_vaccinated,people_vaccinated_per_hundred,people_fully_vaccinated,people_fully_vaccinated_per_hundred,daily_vaccinations
count,42795,2.345700e+04,2.237100e+04,22371.000000,1.946200e+04,19462.000000,4.255800e+04
mean,2021-05-21 12:12:46.182965248,1.869182e+07,8.018487e+06,25.036586,4.958000e+06,18.165081,1.298099e+05
min,2020-12-02 00:00:00,0.000000e+00,0.000000e+00,0.000000,1.000000e+00,0.000000,0.000000e+00
25%,2021-04-02 00:00:00,1.907310e+05,1.555315e+05,3.715000,7.318550e+04,2.110000,8.642500e+02
50%,2021-05-26 00:00:00,1.347062e+06,9.394600e+05,16.700000,5.620635e+05,9.355000,6.911500e+03
75%,2021-07-15 00:00:00,7.128593e+06,4.346202e+06,43.340000,2.825012e+06,28.797500,4.125600e+04
max,2021-09-06 00:00:00,2.113083e+09,1.072500e+09,117.710000,8.894390e+08,116.440000,2.242429e+07
std,NaN,1.074242e+08,3.003110e+07,23.724454,1.777390e+07,20.371328,8.886442e+05


In [10]:
covid_df = covid_data.merge(
    vaccinations_data,
    left_on=['date', 'country'],
    right_on=['date', 'country'],
    how='left'
)
display(covid_df)

,date,country,confirmed,deaths,recovered,active,daily_confirmed,daily_deaths,daily_recovered,total_vaccinations,people_vaccinated,people_vaccinated_per_hundred,people_fully_vaccinated,people_fully_vaccinated_per_hundred,daily_vaccinations,vaccines
0,2020-02-24,Afghanistan,1.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-02-25,Afghanistan,1.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-02-26,Afghanistan,1.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-02-27,Afghanistan,1.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-02-28,Afghanistan,1.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86780,2020-03-12,occupied Palestinian territory,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86781,2020-03-14,occupied Palestinian territory,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86782,2020-03-15,occupied Palestinian territory,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86783,2020-03-16,occupied Palestinian territory,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
covid_df['death_rate'] = (covid_df['deaths']/covid_df['confirmed']) * 100
covid_df['recover_rate'] = (covid_df['recovered']/covid_df['confirmed']) * 100

mask = covid_df['country'] == 'United States'
sorted_us = covid_df[mask].sort_values(by='death_rate', ascending=False)
covid_df.to_csv('data/covid_df.csv', index=False)
display(sorted_us)

,date,country,confirmed,deaths,recovered,active,daily_confirmed,daily_deaths,daily_recovered,total_vaccinations,people_vaccinated,people_vaccinated_per_hundred,people_fully_vaccinated,people_fully_vaccinated_per_hundred,daily_vaccinations,vaccines,death_rate,recover_rate
82584,2020-03-04,United States,153.0,11.0,8.0,134.0,31.0,4.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.189542,5.228758
82656,2020-05-15,United States,1444045.0,87982.0,250747.0,1105316.0,25117.0,1625.0,4333.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.092746,17.364210
82655,2020-05-14,United States,1418928.0,86357.0,246414.0,1086157.0,27013.0,1795.0,2984.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.086073,17.366209
82654,2020-05-13,United States,1391915.0,84562.0,243430.0,1063923.0,20723.0,1700.0,13143.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.075227,17.488855
82657,2020-05-16,United States,1469104.0,89191.0,268376.0,1111537.0,25059.0,1209.0,17629.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.071115,18.268006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82563,2020-02-12,United States,13.0,0.0,3.0,10.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,23.076923
82564,2020-02-13,United States,15.0,0.0,3.0,12.0,2.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,20.000000
82565,2020-02-14,United States,15.0,0.0,3.0,12.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,20.000000
82566,2020-02-15,United States,15.0,0.0,3.0,12.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,20.000000


In [12]:
mask2 = covid_df['country'] == 'Russia'
mean_russia = covid_df[mask2]['recover_rate'].mean()
display(mean_russia)

67.06273489477655